# 02 — Cross-attention testbed

This notebook runs the architecture-specific experiments from Section 5 on a
single 4-head cross-attention layer trained on a *selective copying* task.
This is the first non-synthetic testbed: the value-scale gauge is locally
realized by scaling `W_V`; the task loss is cross-entropy, so curvature is
non-trivial; and the experiment explicitly compares where the gauge is fixed.

The key distinction is now:

- **pre-W_O / head-space**: fix each head aggregate before output mixing,
  `e_h=m_h z_h/||z_h||`, then apply `W_O`;
- **pre-W_O pullback metric**: keep the same head-space gauge fix but penalize
  `||W_O e||²`, matching residual-space energy;
- **post-W_O / residual-space**: fix the gauge after output mixing, so `m` is
  the norm of the actual attention update seen by the readout.

| Experiment | Tests |
|------------|-------|
| 5.1 | Δ_gauge(c) ≈ 0 under `(a,z) → (c⁻¹a,c·z)`; factor-only drift |
| 5.1b | factor-only compensation under a λ sweep |
| 5.2 | λ calibration across head-space, pullback, post-W_O, and controls |
| 5.3 | full-Hessian, capacity-projected, and stationary residual diagnostics |

All sanity checks report bootstrap 95% CIs over seeds. Storage layout matches
01_synthetic and is loaded by 05_analysis: `<ROOT>/<testbed>/runs/*.pkl`.


## Setup

In [ ]:
# Load gauge_lib in either local Jupyter or a Colab-hosted runtime.
# Prefer a local/uploaded gauge_lib.py so notebook results are tied to the
# exact experiment code checked in with this notebook. Fall back to the
# package or GitHub only if no local file is found.
import hashlib
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

GITHUB_PACKAGE = "git+https://github.com/rblicht/Gauge.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def import_fresh_gauge_lib():
    sys.modules.pop("gauge_lib", None)
    return importlib.import_module("gauge_lib")


def install_gauge_lib_from_github():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", GITHUB_PACKAGE])
    module = import_fresh_gauge_lib()
    print(f"Loaded gauge_lib from {Path(module.__file__).resolve()}")
    return module


def load_gauge_lib_from_file(gauge_lib_path: Path):
    gauge_lib_path = gauge_lib_path.resolve()
    sys.path.insert(0, str(gauge_lib_path.parent))
    sys.modules.pop("gauge_lib", None)
    spec = importlib.util.spec_from_file_location("gauge_lib", gauge_lib_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules["gauge_lib"] = module
    spec.loader.exec_module(module)
    print(f"Loaded gauge_lib from {gauge_lib_path}")
    return module


LOCAL_GAUGE_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/mnt/data"),          # ChatGPT / notebook artifact workspace
    Path("/content"),           # Colab upload location
    Path.home() / "Desktop" / "neurips",
    Path("/Users/ryanbruggeman/Desktop/neurips"),
]

local_gauge_path = next(
    (root / "gauge_lib.py" for root in LOCAL_GAUGE_ROOT_CANDIDATES if (root / "gauge_lib.py").exists()),
    None,
)

if local_gauge_path is not None:
    gauge_lib = load_gauge_lib_from_file(local_gauge_path)
elif IN_COLAB:
    try:
        gauge_lib = install_gauge_lib_from_github()
    except Exception as exc:
        print(f"GitHub install failed: {exc}")
        print("If the repo is private, upload gauge_lib.py as a temporary fallback.")
        from google.colab import files
        uploaded = files.upload()
        if "gauge_lib.py" not in uploaded:
            raise FileNotFoundError("Expected an uploaded file named gauge_lib.py")
        gauge_lib = load_gauge_lib_from_file(Path("/content/gauge_lib.py"))
else:
    gauge_lib = import_fresh_gauge_lib()
    print(f"Loaded gauge_lib from {Path(gauge_lib.__file__).resolve()}")

GAUGE_LIB_PATH = Path(gauge_lib.__file__).resolve()
GAUGE_LIB_SHA256 = hashlib.sha256(GAUGE_LIB_PATH.read_bytes()).hexdigest()
print(f"gauge_lib sha256: {GAUGE_LIB_SHA256[:12]}...  ({GAUGE_LIB_SHA256})")

In [ ]:
import math
import time
from copy import deepcopy
from collections import defaultdict
from pathlib import Path
from typing import Callable, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from gauge_lib import (
    ALL_MODES,
    Controller, ControllerInfo,
    RunLogger, RunMeta,
    apply_gauge_traversal, delta_gauge,
    cos_d_neg_g, lambda_m_over_g,
    predict_e_lin,
    predict_e_quad_batch,
    solve_nonnegative_quadratic_subspace,
    set_seed,
)

print('gauge_lib modes:', ALL_MODES)


In [ ]:
# Detect Colab vs local; mount Drive if available.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips2026')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd() / 'neurips2026_local'

ROOT.mkdir(parents=True, exist_ok=True)
print(f'IN_COLAB = {IN_COLAB}')
print(f'ROOT     = {ROOT}')

In [ ]:
# Hyperparameters. Adjust if you want a faster smoke test or a fuller sweep.
SEEDS    = [0, 1, 2, 3, 4]
LAM_5_1  = 1e-2
LAMBDAS_5_2 = (1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0)
C_VALUES_5_1 = (1e-2, 1e-1, 1.0, 1e1, 1e2)

# Selective copying task
V = 8        # vocabulary size (0 = filler, 1..V-1 = real)
L = 8        # input sequence length
K = 4        # number of real tokens per example (= number of query positions)

# Architecture
D_MODEL = 16
N_HEADS = 4
D_HEAD  = D_MODEL // N_HEADS  # = 4

# Per-experiment training step counts (Adam lr=3e-3, batch_size=256).
STEPS_5_1 = 500
STEPS_5_1B = 500
STEPS_5_2 = 1200

# V3 calibration experiments.  Each tuple is:
# (spec_name, controller_variant, magnitude_mode, gauge_location, penalty_metric)
#
# gauge_location:
#   pre_wo  — fix each head before W_O: e_h=m_h z_h/||z_h||, y=W_O concat(e_h)
#   post_wo — fix the actual residual-space attention output after W_O: y=m u/||u||
#
# penalty_metric:
#   euclidean — λ||e||² in the coordinate where gauge is fixed
#   pullback  — for pre_wo only, penalize λ||W_O e||² so the energy matches
#               the downstream residual metric.
CALIBRATION_SPECS_5_2 = (
    ("gauge_fixed_pre_head",       "gauge_fixed", "per_head",  "pre_wo",  "euclidean"),
    ("gauge_fixed_query_mlp",      "gauge_fixed", "query_mlp", "pre_wo",  "euclidean"),
    ("gauge_fixed_pre_pullback",   "gauge_fixed", "per_head",  "pre_wo",  "pullback"),
    ("gauge_fixed_post_wo",        "gauge_fixed", "query_mlp", "post_wo", "euclidean"),
    ("raw_disp_pre_head",          "raw_disp",    "per_head",  "pre_wo",  "euclidean"),
    ("dir_norm_pre_head",          "dir_norm",    "per_head",  "pre_wo",  "euclidean"),
)

# Optional factor-only lambda sweep. This is useful for diagnosing whether
# factor penalties compensate through the unpenalized gauge coordinate.
FACTOR_SWEEP_SPECS_5_1B = ("amp_only", "agg_only", "raw_disp", "gauge_fixed")
LAMBDAS_5_1B = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)

# Optional fixed-backbone probe. This is off by default because it adds another
# sweep; turn it on to test the clean inverse-λ law with representation geometry
# held approximately fixed.
RUN_FIXED_BACKBONE_PROBE = False
LAM_FIXED_BACKBONE_REF = 3e-2
STEPS_FIXED_BACKBONE_MAG = 500

# Set to True to overwrite existing v3 runs. Leaving this False avoids
# accidentally mixing new diagnostics with old notebooks.
FORCE_RERUN = False


### Statistical helpers

All confidence intervals below are computed over **seed-level summaries**
rather than over individual held-out examples. This keeps the uncertainty
estimate aligned with the source of experimental variability: random
initialization and data-generation seed. Conventions match `01_synthetic`.

In [ ]:
def bootstrap_ci_mean(values, ci: float = 0.95, n_boot: int = 10000, seed: int = 0):
    """Mean and percentile bootstrap CI over seed-level values."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    mean = float(vals.mean())
    if vals.size == 1:
        return mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = vals[rng.integers(0, vals.size, size=(n_boot, vals.size))].mean(axis=1)
    alpha = (1.0 - ci) / 2.0
    lo, hi = np.quantile(boot, [alpha, 1.0 - alpha])
    return mean, float(lo), float(hi)


def ci_row(label: str, values, ci: float = 0.95, precision: int = 3):
    mean, lo, hi = bootstrap_ci_mean(values, ci=ci)
    if np.isnan(lo):
        formatted = f"{mean:.{precision}g}"
    else:
        formatted = f"{mean:.{precision}g} [{lo:.{precision}g}, {hi:.{precision}g}]"
    return {"metric": label, "mean": mean, "ci_low": lo, "ci_high": hi, "formatted": formatted}


def display_ci_table(rows, title: str | None = None):
    df = pd.DataFrame(rows)
    if title:
        print(title)
    display(df)
    return df


def per_seed_group(runs):
    grouped = defaultdict(list)
    for r in runs:
        grouped[r["meta"]["seed"]].append(r)
    return grouped

## Helpers — data, loss, training, eval diagnostics

In [ ]:
def make_selective_copy_data(n_train: int = 1024, n_eval: int = 256,
                             V: int = V, L: int = L, K: int = K,
                             seed: int = 0):
    """Generate (input, target) for selective copying.

    Each input has exactly K real tokens (sampled from {1..V-1}) at K random
    positions; the rest are filler (token 0). Positions are sorted ascending
    so the target — the K real tokens in input order — is well defined.
    """
    g = torch.Generator().manual_seed(seed)

    def gen(n: int):
        real = torch.randint(1, V, (n, K), generator=g)
        positions = torch.stack([
            torch.sort(torch.randperm(L, generator=g)[:K]).values
            for _ in range(n)
        ])
        x = torch.zeros(n, L, dtype=torch.long)
        x.scatter_(1, positions, real)
        return x, real

    x_train, t_train = gen(n_train)
    x_eval,  t_eval  = gen(n_eval)
    return {"x_train": x_train, "t_train": t_train,
            "x_eval":  x_eval,  "t_eval":  t_eval}

In [ ]:
def task_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Cross-entropy summed over K query positions, mean over batch."""
    B, K_q, V_ = logits.shape
    return F.cross_entropy(logits.reshape(B * K_q, V_), target.reshape(B * K_q),
                           reduction="sum") / B


def train_attention(model: nn.Module, data: Dict[str, torch.Tensor],
                    steps: int, lr: float = 3e-3, log_every: int = 100,
                    batch_size: int = 256) -> List[Dict]:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    log = []
    x_full, t_full = data["x_train"], data["t_train"]
    n_train = x_full.shape[0]
    for step in range(steps):
        idx = torch.randint(0, n_train, (batch_size,))
        x, t = x_full[idx], t_full[idx]
        logits, info, _ = model(x)
        L_task = task_loss(logits, t)
        L = L_task + info.penalty
        opt.zero_grad()
        L.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            with torch.no_grad():
                pred = logits.argmax(dim=-1)
                acc = (pred == t).float().mean().item()
                log.append({
                    "step": step, "loss": L.item(),
                    "task_loss": L_task.item(),
                    "penalty": info.penalty.item(),
                    "acc": acc,
                    "a_mean": info.a.mean().item(),
                    "m_mean": info.m.mean().item(),
                    "z_norm_mean": info.z_norm.mean().item(),
                    "e_norm_mean": info.e_norm.mean().item(),
                })
    return log

### Eval diagnostics — closed-form \(g\), full \(H\), and residuals at the reference point \(e=0\)

For this single-attention-layer architecture, \(e=0\) means the attention
output contribution is set to zero before `W_O` and the readout.  Because the
readout is linear, the task gradient and Hessian with respect to the
concatenated displacement are available in closed form.  We use the full
concatenated Hessian rather than a per-head block approximation, then reshape
the prediction back into heads for per-head controller diagnostics.


In [ ]:
def _attention_geometry(model, t_eval, coordinate: str, e_obs):
    """Return g, H, metric, and target geometry in the displacement coordinate.

    coordinate='pre_wo' uses the concatenated pre-output attention displacement.
    coordinate='post_wo' uses the post-W_O residual-space attention output.
    """
    V_ = model.V
    readout_W = model.readout.weight                    # (V, D)
    p0 = F.softmax(model.readout.bias, dim=-1)           # logits at zero displacement
    target_onehot = F.one_hot(t_eval, num_classes=V_).float()
    diff = p0[None, None, :] - target_onehot             # (B,K,V)
    H_pp = torch.diag(p0) - torch.outer(p0, p0)           # (V,V)

    if coordinate == "pre_wo":
        W_O = model.attention.W_O.weight                 # (D,D)
        J = readout_W @ W_O                              # logits = readout(W_O e)
        metric = W_O.T @ W_O if model.attention.penalty_metric == "pullback" else None
    elif coordinate == "post_wo":
        J = readout_W                                    # logits = readout(e)
        metric = None
    else:
        raise ValueError(f"unknown coordinate={coordinate!r}")

    g = diff @ J                                         # (B,K,D)
    H = J.T @ H_pp @ J                                   # (D,D)
    return g, H, metric, target_onehot


def _embed_head_directions(d_pre):
    """Convert per-head directions (B,H,K,d_head) to row directions (B,K,H,D)."""
    B, H_, K_q, d_head = d_pre.shape
    D = H_ * d_head
    d_bkhd = d_pre.transpose(1, 2)                       # (B,K,H,d_head)
    dirs = torch.zeros(B, K_q, H_, D, device=d_pre.device, dtype=d_pre.dtype)
    for h in range(H_):
        dirs[:, :, h, h*d_head:(h+1)*d_head] = d_bkhd[:, :, h, :]
    return dirs


def _full_solve(A, g):
    """Solve A e = -g for g shape (B,K,D)."""
    B, K_q, D = g.shape
    rhs = (-g).reshape(B * K_q, D).T
    return torch.linalg.solve(A, rhs).T.reshape(B, K_q, D)


def eval_diagnostics_attention(model, data, lam):
    """Diagnostics for whichever displacement coordinate the model controls.

    pre_wo models return per-head arrays for m/cos/targets and concatenated
    residuals for r_lin/r_quad/r_subspace.  post_wo models return per-query
    arrays because the controlled displacement is already residual-space.
    """
    model.eval()
    x_eval, t_eval = data["x_eval"], data["t_eval"]
    B = x_eval.shape[0]
    H_ = model.attention.n_heads
    d_head = model.attention.d_head
    K_q = model.K
    D = model.attention.d_model
    eps = 1e-12

    with torch.no_grad():
        logits, info, e_obs = model(x_eval)
        coordinate = model.attention.gauge_location
        g_concat, H_concat, metric, target_onehot = _attention_geometry(model, t_eval, coordinate, e_obs)

        G = torch.eye(D, device=H_concat.device, dtype=H_concat.dtype) if metric is None else metric.to(H_concat)
        A_lin = 2.0 * lam * G
        A_quad = H_concat + 2.0 * lam * G
        # Tiny ridge protects against a singular pullback metric if W_O is poorly conditioned.
        ridge = 1e-8 * torch.eye(D, device=H_concat.device, dtype=H_concat.dtype)
        e_lin_concat = _full_solve(A_lin + ridge, g_concat)
        e_quad_concat = _full_solve(A_quad + ridge, g_concat)

        # Gradient at the observed displacement for stationary residual.
        p_obs = F.softmax(logits, dim=-1)
        diff_obs = p_obs - target_onehot
        if coordinate == "pre_wo":
            J = model.readout.weight @ model.attention.W_O.weight
            grad_obs_concat = diff_obs @ J
            e_obs_concat = e_obs.transpose(1, 2).reshape(B, K_q, D)
            d_for_target = info.d                              # (B,H,K,d_head)
            dirs = _embed_head_directions(d_for_target)         # (B,K,H,D)
            m_sub, e_sub_concat, _ = solve_nonnegative_quadratic_subspace(
                g_concat, H_concat, dirs, lam, penalty_metric=metric, nonnegative=True
            )                                                   # m_sub: (B,K,H)
            m_sub_target = m_sub.transpose(1, 2)                 # (B,H,K)

            e_lin_per_head = e_lin_concat.reshape(B, K_q, H_, d_head).transpose(1, 2)
            e_quad_per_head = e_quad_concat.reshape(B, K_q, H_, d_head).transpose(1, 2)
            g_per_head = g_concat.reshape(B, K_q, H_, d_head).transpose(1, 2)

            m_lin_target = e_lin_per_head.norm(dim=-1)
            m_quad_target = e_quad_per_head.norm(dim=-1)
            g_norm = g_per_head.norm(dim=-1)
            cosv = cos_d_neg_g(info.d, g_per_head)
            r_subspace = (e_obs_concat - e_sub_concat).norm(dim=-1)

        else:  # post_wo: e_obs is already the residual-space displacement (B,K,D)
            J = model.readout.weight
            grad_obs_concat = diff_obs @ J
            e_obs_concat = e_obs
            dirs = info.d.unsqueeze(-2)                         # (B,K,1,D)
            m_sub, e_sub_concat, _ = solve_nonnegative_quadratic_subspace(
                g_concat, H_concat, dirs, lam, penalty_metric=None, nonnegative=True
            )
            m_sub_target = m_sub.squeeze(-1)                     # (B,K)
            m_lin_target = e_lin_concat.norm(dim=-1)
            m_quad_target = e_quad_concat.norm(dim=-1)
            g_norm = g_concat.norm(dim=-1)
            cosv = cos_d_neg_g(info.d, g_concat)
            r_subspace = (e_obs_concat - e_sub_concat).norm(dim=-1)

        grad_energy = torch.einsum("ij,bkj->bki", G, e_obs_concat)
        stat_vec = grad_obs_concat + 2.0 * lam * grad_energy
        r_stat = stat_vec.norm(dim=-1)
        r_stat_norm = r_stat / (
            grad_obs_concat.norm(dim=-1) + 2.0 * lam * grad_energy.norm(dim=-1) + eps
        )

        r_lin = (e_obs_concat - e_lin_concat).norm(dim=-1)
        r_quad = (e_obs_concat - e_quad_concat).norm(dim=-1)

        pred = logits.argmax(dim=-1)
        acc = (pred == t_eval).float().mean().item()

        # Energy-space norm of observed displacement.  For pullback pre_wo this
        # is ||W_O e||; otherwise it is the coordinate Euclidean norm.
        e_energy_norm = torch.sqrt((e_obs_concat * torch.einsum("ij,bkj->bki", G, e_obs_concat)).sum(dim=-1).clamp_min(0.0))

        return {
            "coordinate": coordinate,
            "penalty_metric": model.attention.penalty_metric,
            "a": info.a.detach().cpu().numpy(),
            "z_norm": info.z_norm.detach().cpu().numpy(),
            "e_norm": info.e_norm.detach().cpu().numpy(),
            "e_energy_norm": e_energy_norm.detach().cpu().numpy(),
            "m": info.m.detach().cpu().numpy(),
            "g_norm": g_norm.detach().cpu().numpy(),
            "m_lin_target": m_lin_target.detach().cpu().numpy(),
            "m_quad_target": m_quad_target.detach().cpu().numpy(),
            "m_subspace_target": m_sub_target.detach().cpu().numpy(),
            "cos_d_neg_g": cosv.detach().cpu().numpy(),
            "r_lin": r_lin.detach().cpu().numpy(),
            "r_quad": r_quad.detach().cpu().numpy(),
            "r_subspace": r_subspace.detach().cpu().numpy(),
            "r_stat": r_stat.detach().cpu().numpy(),
            "r_stat_norm": r_stat_norm.detach().cpu().numpy(),
            "task_metric": acc,
        }


## Architecture

`GaugedCrossAttention` exposes the value-aggregate gauge through the value
projection `W_V`.  The gauge intervention scales `W_V` by `c` and scales the
amplitude channel by `c^{-1}` for amplitude-based variants.  For
`gauge_fixed`, the scalar is the calibrated magnitude `m`, so amplitude scaling
is a no-op and scaling `W_V` only changes the raw norm, not the normalized
direction.

The original controller uses one scalar per head (`magnitude_mode="per_head"`).
This is intentionally capacity-limited: it can shrink with λ but cannot track
per-example or per-query gradient norms.  The improved variant
(`magnitude_mode="query_mlp"`) predicts a positive scalar from query/key
features that are independent of `W_V`, preserving the value-scale gauge while
allowing pointwise magnitude calibration.


In [ ]:
class GaugedCrossAttention(nn.Module):
    """Single cross-attention layer with an explicit scalar gauge channel.

    gauge_location:
      - "pre_wo":  fix the gauge in per-head value-aggregate coordinates.
        This is a head-space diagnostic: e_h=m_h z_h/||z_h||, then W_O mixes heads.
      - "post_wo": fix the gauge after W_O, so m is the norm of the actual
        residual-space attention update seen by the readout.

    magnitude_mode:
      - "per_head":  one learned scalar per head (pre_wo only).
      - "query_mlp": positive scalar per query; pre_wo gives one per head/query,
        post_wo gives one per query.  Features are query/key-context only, so
        the magnitude channel is independent of value-scale gauge traversal.

    penalty_metric:
      - "euclidean": λ||e||² in the coordinate where gauge is fixed.
      - "pullback": pre_wo only; use λ||W_O e||² to match residual-space energy.
    """
    def __init__(self, d_model: int, n_heads: int, mode: str, lam: float,
                 fixed_magnitude: float = 1.0, init_log_scale: float = 0.0,
                 magnitude_mode: str = "per_head", magnitude_hidden: int = 16,
                 gauge_location: str = "pre_wo", penalty_metric: str = "euclidean"):
        super().__init__()
        assert d_model % n_heads == 0
        if gauge_location not in ("pre_wo", "post_wo"):
            raise ValueError(f"unknown gauge_location={gauge_location!r}")
        if penalty_metric not in ("euclidean", "pullback"):
            raise ValueError(f"unknown penalty_metric={penalty_metric!r}")
        if penalty_metric == "pullback" and gauge_location != "pre_wo":
            raise ValueError("pullback metric is only defined for pre_wo gauge fixing")
        if magnitude_mode not in ("per_head", "query_mlp"):
            raise ValueError(f"unknown magnitude_mode={magnitude_mode!r}")
        if gauge_location == "post_wo" and magnitude_mode == "per_head":
            raise ValueError("post_wo gauge fixing needs query_mlp magnitude_mode")

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.magnitude_mode = magnitude_mode
        self.gauge_location = gauge_location
        self.penalty_metric = penalty_metric
        self.lam = float(lam)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        # Global multiplicative scalar.  In pre_wo it is one scalar per head;
        # in post_wo it is one scalar per residual-space query update.
        scalar_shape = (n_heads,) if gauge_location == "pre_wo" else (1,)
        self.scalar_log_b = nn.Parameter(torch.full(scalar_shape, init_log_scale))

        if magnitude_mode == "query_mlp":
            feat_dim = 2 * (self.d_head if gauge_location == "pre_wo" else d_model)
            self.magnitude_net = nn.Sequential(
                nn.Linear(feat_dim, magnitude_hidden),
                nn.Tanh(),
                nn.Linear(magnitude_hidden, 1),
            )
            nn.init.zeros_(self.magnitude_net[-1].weight)
            nn.init.zeros_(self.magnitude_net[-1].bias)
        else:
            self.magnitude_net = None

        # In pre_wo, the theoretical per-query energy is λ∑_h m_{h}² averaged
        # over batch/query positions, not λ mean_{h} m_h².  The same structured
        # reduction is used for factor penalties so comparisons share scale.
        penalty_sum_dims = (1,) if gauge_location == "pre_wo" else None
        self.controller = Controller(
            mode=mode, lam=lam, fixed_magnitude=fixed_magnitude,
            penalty_sum_dims=penalty_sum_dims,
        )

    def _split_heads(self, x):
        B, T, _ = x.shape
        return x.reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)

    def _scalar_channel_pre(self, Q, K_context):
        B, H_, K_q, _ = Q.shape
        scale = torch.exp(self.scalar_log_b).view(1, H_, 1)
        if self.magnitude_mode == "per_head":
            base = torch.ones(B, H_, K_q, device=Q.device, dtype=Q.dtype)
        else:
            features = torch.cat([Q, K_context], dim=-1)
            base = F.softplus(self.magnitude_net(features).squeeze(-1)) + 1e-6
        return scale * base

    def _scalar_channel_post(self, Q, K_context):
        # Q/K_context are concatenated residual-space features with shape (B,K,D).
        scale = torch.exp(self.scalar_log_b).view(1, 1)
        features = torch.cat([Q, K_context], dim=-1)
        base = F.softplus(self.magnitude_net(features).squeeze(-1)) + 1e-6
        return scale * base

    def forward(self, queries, keys_values):
        B, K_q, _ = queries.shape

        Q = self._split_heads(self.W_Q(queries))
        K_proj = self._split_heads(self.W_K(keys_values))
        V_proj = self._split_heads(self.W_V(keys_values))

        attn_scores = (Q @ K_proj.transpose(-1, -2)) / math.sqrt(self.d_head)
        attn_weights = F.softmax(attn_scores, dim=-1)
        z = attn_weights @ V_proj                              # (B,H,K,d_head)
        K_context = attn_weights @ K_proj                      # value-scale invariant

        if self.gauge_location == "pre_wo":
            s = self._scalar_channel_pre(Q, K_context)          # (B,H,K)
            e, info = self.controller(z, s)                     # (B,H,K,d_head)
            e_concat = e.transpose(1, 2).reshape(B, K_q, self.d_model)
            output = self.W_O(e_concat)

            if self.penalty_metric == "pullback" and self.controller.mode in ("gauge_fixed", "raw_disp"):
                # Replace λ||e||² with λ||W_O e||², averaged over batch/query.
                info.penalty = self.lam * output.norm(dim=-1).pow(2).mean()
            return output, info, e

        # post_wo: first form the residual-space raw displacement u=W_O concat(z),
        # then fix the gauge and return e directly to the readout.
        z_concat = z.transpose(1, 2).reshape(B, K_q, self.d_model)
        raw_output = self.W_O(z_concat)
        Q_concat = Q.transpose(1, 2).reshape(B, K_q, self.d_model)
        K_context_concat = K_context.transpose(1, 2).reshape(B, K_q, self.d_model)
        s = self._scalar_channel_post(Q_concat, K_context_concat)  # (B,K)
        e_post, info = self.controller(raw_output, s)              # (B,K,D)
        return e_post, info, e_post

    @torch.no_grad()
    def scale_amp(self, factor: float) -> None:
        if factor <= 0:
            raise ValueError(f"factor must be positive, got {factor}")
        if self.controller.mode in ("gauge_fixed", "dir_norm"):
            return
        self.scalar_log_b.add_(math.log(factor))

    @torch.no_grad()
    def scale_agg(self, factor: float) -> None:
        # Scaling values scales z and therefore also W_O concat(z); attention
        # weights and query/key magnitude features are unchanged.
        self.W_V.weight.mul_(factor)


In [ ]:
class SelectiveCopyModel(nn.Module):
    def __init__(self, V: int, L: int, K: int, d_model: int, n_heads: int,
                 mode: str, lam: float, fixed_magnitude: float = 1.0,
                 magnitude_mode: str = "per_head", gauge_location: str = "pre_wo",
                 penalty_metric: str = "euclidean"):
        super().__init__()
        self.V, self.L, self.K = V, L, K
        self.token_embed = nn.Embedding(V, d_model)
        self.pos_embed   = nn.Embedding(L, d_model)
        self.queries     = nn.Parameter(torch.randn(K, d_model) * 0.1)

        self.attention = GaugedCrossAttention(
            d_model=d_model, n_heads=n_heads, mode=mode, lam=lam,
            fixed_magnitude=fixed_magnitude,
            magnitude_mode=magnitude_mode,
            gauge_location=gauge_location,
            penalty_metric=penalty_metric,
        )
        self.readout = nn.Linear(d_model, V)
        self.mode, self.lam = mode, lam
        self.magnitude_mode = magnitude_mode
        self.gauge_location = gauge_location
        self.penalty_metric = penalty_metric

    def forward(self, x):
        B = x.shape[0]
        positions = torch.arange(self.L, device=x.device)
        x_embed = self.token_embed(x) + self.pos_embed(positions)[None, :, :]
        queries = self.queries[None, :, :].expand(B, -1, -1)
        attn_out, info, e_coord = self.attention(queries, x_embed)
        logits = self.readout(attn_out)
        return logits, info, e_coord

    def scale_amp(self, factor: float) -> None: self.attention.scale_amp(factor)
    def scale_agg(self, factor: float) -> None: self.attention.scale_agg(factor)


## Experiment 5.1 — Gauge traversal and factor-only drift

Train all six controller variants at fixed λ. After training, apply
`(a, z) → (c⁻¹a, c·z)` for `c ∈ {1e-2, 1e-1, 1, 1e1, 1e2}` (uniform across
heads) and measure `Δ_gauge(c) = ‖e_after − e_before‖ / ‖e_before‖`.

**Prediction**: Δ_gauge ≈ 0 (numerical floor) across all variants — the
attention layer admits the (a, z) gauge by construction, and softmax
attention weights are independent of `V`-scale.

The factor-only drift signature for `amp_only` and `agg_only` is documented
in the analysis notebook; here we just save the per-step training-log values
of `a_mean`, `z_norm_mean`, `e_norm_mean` so they can be inspected.

In [ ]:
def run_experiment_5_1(root: Path, lam: float, seeds: List[int], c_values, steps: int):
    print(f"=== 5.1 attention gauge traversal (lam={lam}) ===")
    t0 = time.time()
    for seed in seeds:
        for variant in ALL_MODES:
            set_seed(seed)
            data = make_selective_copy_data(seed=seed)
            model = SelectiveCopyModel(V=V, L=L, K=K, d_model=D_MODEL,
                                        n_heads=N_HEADS, mode=variant, lam=lam, magnitude_mode="per_head")
            train_log = train_attention(model, data, steps=steps, log_every=50)

            with torch.no_grad():
                _, _, e_before = model(data["x_eval"])

            traversal = {}
            for c in c_values:
                m_copy = deepcopy(model)
                apply_gauge_traversal(c, m_copy.scale_amp, m_copy.scale_agg)
                with torch.no_grad():
                    _, info_after, e_after = m_copy(data["x_eval"])
                rel = (e_after - e_before).norm(dim=-1) / e_before.norm(dim=-1).clamp_min(1e-12)
                traversal[float(c)] = {
                    "delta_gauge_mean": rel.mean().item(),
                    "delta_gauge_max":  rel.max().item(),
                    "a_mean":      info_after.a.mean().item(),
                    "z_norm_mean": info_after.z_norm.mean().item(),
                    "e_norm_mean": info_after.e_norm.mean().item(),
                }

            diag = eval_diagnostics_attention(model, data, lam)
            meta = RunMeta(testbed="attention_5_1", variant=variant, lam=lam, seed=seed)
            logger = RunLogger(root, meta, config={
                "experiment": "5_1", "lam": lam, "seed": seed,
                "V": V, "L": L, "K": K, "d_model": D_MODEL, "n_heads": N_HEADS,
                "steps": steps, "magnitude_mode": "per_head",
                "gauge_lib_path": str(GAUGE_LIB_PATH),
                "gauge_lib_sha256": GAUGE_LIB_SHA256,
            })
            for row in train_log:
                logger.log_step(**row)
            logger.update_final(**diag)
            logger.set_final("gauge_traversal", traversal)
            logger.save()
        print(f"  seed={seed} done")
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_1(ROOT, LAM_5_1, SEEDS, C_VALUES_5_1, STEPS_5_1)


**Sanity-check plots and tables**: max-Δ_gauge across c by variant, plus a seed-level factor-only drift summary. All Δ_gauge bars should be at numerical floor (≲ 1e-6).

In [ ]:
# 5.1 sanity check: gauge traversal + factor-only drift summary
runs = RunLogger.load_testbed(ROOT, "attention_5_1")
delta_max_by_variant = defaultdict(list)
for r in runs:
    v = r["meta"]["variant"]
    for c, stats in r["final"]["gauge_traversal"].items():
        delta_max_by_variant[v].append(stats["delta_gauge_max"])

variants_order = list(ALL_MODES)
max_deltas = [max(delta_max_by_variant[v]) for v in variants_order]

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(variants_order)), max_deltas, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(range(len(variants_order)))
ax.set_xticklabels(variants_order, rotation=15)
ax.set_ylabel("max Δ_gauge across c, seeds")
ax.axhline(1e-6, color="red", linestyle="--", linewidth=1, label="numerical floor")
ax.set_title("5.1 — gauge traversal residuals (cross-attention)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nWorst-case Δ_gauge per variant:")
for v, d in zip(variants_order, max_deltas):
    print(f"  {v:>14}: {d:.2e}")

# Seed-level CIs of max Δ_gauge per variant.
ci_rows = []
for v in variants_order:
    seed_vals = []
    for seed in sorted({r["meta"]["seed"] for r in runs if r["meta"]["variant"] == v}):
        rs = [r for r in runs if r["meta"]["variant"] == v and r["meta"]["seed"] == seed]
        vals = []
        for r in rs:
            vals.extend(stats["delta_gauge_max"] for stats in r["final"]["gauge_traversal"].values())
        seed_vals.append(max(vals))
    row = ci_row(v, seed_vals, precision=3)
    row["n_seeds"] = len(seed_vals)
    ci_rows.append(row)

delta_ci_df = pd.DataFrame(ci_rows).rename(columns={"metric": "variant", "formatted": "max Δ_gauge mean [95% CI]"})
display(delta_ci_df[["variant", "n_seeds", "mean", "ci_low", "ci_high", "max Δ_gauge mean [95% CI]"]])

# Factor-only drift: final / initial training-log values, summarized over seeds.
drift_rows = []
for v in variants_order:
    for metric in ["a_mean", "z_norm_mean", "e_norm_mean"]:
        ratios = []
        finals = []
        initials = []
        for r in runs:
            if r["meta"]["variant"] != v:
                continue
            log = r.get("training_log", [])
            if not log or metric not in log[0] or metric not in log[-1]:
                continue
            init  = float(log[0][metric])
            final = float(log[-1][metric])
            initials.append(init)
            finals.append(final)
            ratios.append(final / max(abs(init), 1e-12))
        if ratios:
            mean_ratio, lo_ratio, hi_ratio = bootstrap_ci_mean(ratios)
            mean_init, _, _ = bootstrap_ci_mean(initials)
            mean_final, _, _ = bootstrap_ci_mean(finals)
            drift_rows.append({
                "variant": v, "metric": metric,
                "initial_mean": mean_init, "final_mean": mean_final,
                "final/initial_mean": mean_ratio,
                "final/initial_ci_low": lo_ratio,
                "final/initial_ci_high": hi_ratio,
                "n_seeds": len(ratios),
            })

drift_df = pd.DataFrame(drift_rows)
print("\nFactor-only drift summary (seed-level bootstrap CIs for final / initial):")
display(drift_df)

# Compact view for the main gauge-pathology variants.
main_drift = drift_df[drift_df["variant"].isin(["amp_only", "agg_only", "raw_disp", "gauge_fixed"])]
print("\nCompact drift view:")
display(main_drift.pivot(index="variant", columns="metric", values="final/initial_mean"))

## Experiment 5.1b — factor-only λ sweep

The fixed-λ drift table above can miss the strongest pathology if the factor
penalty is too weak.  This optional sweep trains the factor-only variants across
λ and summarizes whether the penalized coordinate is controlled by compensating
movement in the unpenalized coordinate.  It is diagnostic rather than part of
the main calibration claim.


In [ ]:
def run_experiment_5_1b(root: Path, lambdas, seeds: List[int], variants, steps: int,
                        force: bool = FORCE_RERUN):
    print("=== 5.1b attention factor-only lambda sweep ===")
    t0 = time.time()
    for variant in variants:
        testbed = f"attention_5_1b_{variant}"
        for seed in seeds:
            for lam in lambdas:
                meta = RunMeta(testbed=testbed, variant=variant, lam=float(lam), seed=seed,
                               extra={"experiment": "5_1b", "magnitude_mode": "per_head"})
                logger = RunLogger(root, meta, config={
                    "experiment": "5_1b", "variant": variant, "lam": float(lam), "seed": seed,
                    "steps": steps, "magnitude_mode": "per_head",
                    "gauge_location": "pre_wo", "penalty_metric": "euclidean",
                    "gauge_lib_path": str(GAUGE_LIB_PATH),
                    "gauge_lib_sha256": GAUGE_LIB_SHA256,
                })
                if logger.path.exists() and not force:
                    continue

                set_seed(seed)
                data = make_selective_copy_data(seed=seed)
                model = SelectiveCopyModel(V=V, L=L, K=K, d_model=D_MODEL,
                                            n_heads=N_HEADS, mode=variant, lam=float(lam),
                                            magnitude_mode="per_head",
                                            gauge_location="pre_wo",
                                            penalty_metric="euclidean")
                train_log = train_attention(model, data, steps=steps, log_every=100)
                diag = eval_diagnostics_attention(model, data, float(lam))
                for row in train_log:
                    logger.log_step(**row)
                logger.update_final(**diag)
                logger.save()
        print(f"  {variant} done")
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_1b(ROOT, LAMBDAS_5_1B, SEEDS, FACTOR_SWEEP_SPECS_5_1B, STEPS_5_1B)


# Factor-only sweep summary: final/initial ratios and final displacement scale.
sweep_rows = []
for variant in FACTOR_SWEEP_SPECS_5_1B:
    testbed = f"attention_5_1b_{variant}"
    runs = RunLogger.load_testbed(ROOT, testbed)
    for lam in sorted({float(r["meta"]["lam"]) for r in runs}):
        seed_rows = [r for r in runs if float(r["meta"]["lam"]) == lam]
        if not seed_rows:
            continue
        for metric in ["a_mean", "z_norm_mean", "e_norm_mean"]:
            ratios, finals = [], []
            for r in seed_rows:
                log = r.get("training_log", [])
                if not log or metric not in log[0] or metric not in log[-1]:
                    continue
                ratios.append(float(log[-1][metric]) / max(abs(float(log[0][metric])), 1e-12))
                finals.append(float(log[-1][metric]))
            if ratios:
                mean, lo, hi = bootstrap_ci_mean(ratios)
                sweep_rows.append({
                    "variant": variant, "λ": lam, "metric": metric,
                    "final/initial": mean, "ci_low": lo, "ci_high": hi,
                    "final_mean": float(np.mean(finals)), "n_seeds": len(ratios),
                })

factor_sweep_df = pd.DataFrame(sweep_rows)
print("Factor-only λ sweep: final/initial ratios.")
display(factor_sweep_df)

if not factor_sweep_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
    for ax, metric in zip(axes, ["a_mean", "z_norm_mean", "e_norm_mean"]):
        for variant in FACTOR_SWEEP_SPECS_5_1B:
            sub = factor_sweep_df[(factor_sweep_df["variant"] == variant) &
                                  (factor_sweep_df["metric"] == metric)]
            if sub.empty:
                continue
            ax.plot(sub["λ"], sub["final/initial"], "o-", label=variant)
        ax.set_xscale("log")
        ax.set_xlabel("λ")
        ax.set_ylabel("final / initial")
        ax.set_title(metric)
        ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
    axes[-1].legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


## Experiment 5.2 — Controller calibration under λ sweeps

We run λ sweeps for four gauge-fixed coordinate choices and two negative
controls.  The goal is not only to improve magnitude calibration, but to
identify why calibration weakens when the magnitude channel is capacity-limited
or when the gauge is fixed in a coordinate space different from the downstream
residual update.

The main comparisons are:

1. **pre-head**: original per-head gauge fix before `W_O`;
2. **query-MLP**: input/query-dependent magnitudes before `W_O`;
3. **pre-pullback**: per-head gauge fix with the pullback metric `||W_Oe||²`;
4. **post-W_O**: gauge fix after `W_O`, so `m` is the actual residual-space norm;
5. **raw displacement** and **direction-only** controls.

The key comparison is not only `2λm/‖g‖`; in cross-entropy attention the
curvature is not `H=I`, so we report `m/‖e_quad‖` and the new capacity-projected
target `m/m_subspace` in addition to the linearized ratio.


In [ ]:
def run_experiment_5_2(root: Path, lambdas, seeds: List[int], steps: int,
                       specs=CALIBRATION_SPECS_5_2, force: bool = FORCE_RERUN):
    print("=== 5.2 attention lambda sweeps (v3 coordinate diagnostics) ===")
    t0 = time.time()
    for spec_name, variant, magnitude_mode, gauge_location, penalty_metric in specs:
        testbed = f"attention_5_2_v3_{spec_name}"
        print(f"--- {spec_name}: variant={variant}, magnitude_mode={magnitude_mode}, "
              f"gauge_location={gauge_location}, penalty_metric={penalty_metric} ---")
        for seed in seeds:
            for lam in lambdas:
                meta = RunMeta(testbed=testbed, variant=variant, lam=float(lam), seed=seed,
                               extra={"spec_name": spec_name,
                                      "magnitude_mode": magnitude_mode,
                                      "gauge_location": gauge_location,
                                      "penalty_metric": penalty_metric,
                                      "experiment": "5_2_v3"})
                logger = RunLogger(root, meta, config={
                    "experiment": "5_2_v3", "spec_name": spec_name,
                    "variant": variant, "magnitude_mode": magnitude_mode,
                    "gauge_location": gauge_location,
                    "penalty_metric": penalty_metric,
                    "lam": float(lam), "seed": seed, "steps": steps,
                    "V": V, "L": L, "K": K, "d_model": D_MODEL, "n_heads": N_HEADS,
                    "gauge_lib_path": str(GAUGE_LIB_PATH),
                    "gauge_lib_sha256": GAUGE_LIB_SHA256,
                })
                if logger.path.exists() and not force:
                    continue

                set_seed(seed)
                data = make_selective_copy_data(seed=seed)
                model = SelectiveCopyModel(V=V, L=L, K=K, d_model=D_MODEL,
                                            n_heads=N_HEADS, mode=variant, lam=float(lam),
                                            magnitude_mode=magnitude_mode,
                                            gauge_location=gauge_location,
                                            penalty_metric=penalty_metric)
                train_log = train_attention(model, data, steps=steps, log_every=200)
                diag = eval_diagnostics_attention(model, data, float(lam))

                for row in train_log:
                    logger.log_step(**row)
                logger.update_final(**diag)
                logger.save()
            print(f"  seed={seed} done")
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_2(ROOT, LAMBDAS_5_2, SEEDS, STEPS_5_2)


**Sanity-check plots and tables**: compare median magnitude, energy-space
observed displacement norm, gradient norm, target ratios, direction alignment,
and task accuracy across coordinate choices and controls.  For attention, the
most diagnostic quantities are `m/‖e_quad‖`, `m/capacity_target`, and the slope
of `‖g‖` versus λ, since retraining the whole model changes local geometry.


In [ ]:
def load_calibration_runs(specs=CALIBRATION_SPECS_5_2):
    runs_by_spec = {}
    for spec_name, *_ in specs:
        testbed = f"attention_5_2_v3_{spec_name}"
        runs = RunLogger.load_testbed(ROOT, testbed)
        runs_by_spec[spec_name] = runs
    return runs_by_spec


def seed_metric_rows_for_spec(runs):
    """Return one seed-level row per λ/run."""
    rows = []
    for r in runs:
        lam = float(r["meta"]["lam"])
        final = r["final"]
        m = np.asarray(final["m"]).reshape(-1)
        g = np.asarray(final["g_norm"]).reshape(-1)
        m_lin = np.asarray(final["m_lin_target"]).reshape(-1)
        m_quad = np.asarray(final["m_quad_target"]).reshape(-1)
        m_sub = np.asarray(final.get("m_subspace_target", np.nan)).reshape(-1)
        r_lin = np.asarray(final["r_lin"]).reshape(-1)
        r_quad = np.asarray(final["r_quad"]).reshape(-1)
        r_sub = np.asarray(final.get("r_subspace", np.nan)).reshape(-1)
        r_stat = np.asarray(final["r_stat"]).reshape(-1)
        r_stat_norm = np.asarray(final.get("r_stat_norm", np.nan)).reshape(-1)
        cosv = np.asarray(final["cos_d_neg_g"]).reshape(-1)
        e_energy_norm = np.asarray(final.get("e_energy_norm", np.nan)).reshape(-1)
        rows.append({
            "λ": lam,
            "seed": r["meta"]["seed"],
            "coordinate": final.get("coordinate", r["meta"].get("extra", {}).get("gauge_location", "unknown")),
            "penalty_metric": final.get("penalty_metric", r["meta"].get("extra", {}).get("penalty_metric", "unknown")),
            "m_obs": float(np.median(m)),
            "e_energy_norm": float(np.nanmedian(e_energy_norm)),
            "g_norm": float(np.median(g)),
            "m_lin_target": float(np.median(m_lin)),
            "m_quad_target": float(np.median(m_quad)),
            "m_subspace_target": float(np.nanmedian(m_sub)),
            "m_over_m_lin": float(np.median(m / np.maximum(m_lin, 1e-12))),
            "m_over_m_quad": float(np.median(m / np.maximum(m_quad, 1e-12))),
            "m_over_m_subspace": float(np.nanmedian(m / np.maximum(m_sub, 1e-12))),
            "calib_linear": float(np.median(2.0 * lam * m / np.maximum(g, 1e-12))),
            "calib_quadratic_HI": float(np.median((1.0 + 2.0 * lam) * m / np.maximum(g, 1e-12))),
            "cos_d_neg_g": float(np.median(cosv)),
            "r_lin": float(np.median(r_lin)),
            "r_quad": float(np.median(r_quad)),
            "r_subspace": float(np.nanmedian(r_sub)),
            "r_quad_over_r_lin": float(np.median(r_quad) / max(np.median(r_lin), 1e-12)),
            "r_subspace_over_r_quad": float(np.nanmedian(r_sub) / max(np.median(r_quad), 1e-12)),
            "r_stat": float(np.median(r_stat)),
            "r_stat_norm": float(np.nanmedian(r_stat_norm)),
            "task_metric": float(final["task_metric"]),
        })
    return pd.DataFrame(rows)


runs_by_spec = load_calibration_runs()
seed_dfs = {spec: seed_metric_rows_for_spec(runs)
            for spec, runs in runs_by_spec.items() if len(runs) > 0}

if not seed_dfs:
    print("No v3 calibration runs found. Run the 5.2 training cell above.")
else:
    curve_rows = []
    for spec, df in seed_dfs.items():
        for lam, sub in df.groupby("λ"):
            row = {"spec": spec, "λ": lam}
            for key in ["m_obs", "e_energy_norm", "g_norm", "m_over_m_quad", "m_over_m_subspace",
                        "calib_linear", "cos_d_neg_g", "task_metric"]:
                row[key] = float(sub[key].mean())
            curve_rows.append(row)
    curves = pd.DataFrame(curve_rows)

    fig, axes = plt.subplots(2, 4, figsize=(17, 7))

    for spec, sub in curves.groupby("spec"):
        label = spec.replace("_", " ")
        axes[0,0].plot(sub["λ"], sub["m_obs"], "o-", label=label)
        axes[0,1].plot(sub["λ"], sub["e_energy_norm"], "o-", label=label)
        axes[0,2].plot(sub["λ"], sub["g_norm"], "o-", label=label)
        axes[0,3].plot(sub["λ"], sub["m_over_m_quad"], "o-", label=label)
        axes[1,0].plot(sub["λ"], sub["m_over_m_subspace"], "o-", label=label)
        axes[1,1].plot(sub["λ"], sub["calib_linear"], "o-", label=label)
        axes[1,2].plot(sub["λ"], sub["cos_d_neg_g"], "o-", label=label)
        axes[1,3].plot(sub["λ"], sub["task_metric"], "o-", label=label)

    axes[0,0].set_title("observed median m")
    axes[0,1].set_title("observed displacement energy norm")
    axes[0,2].set_title("median ‖g‖ (geometry drift)")
    axes[0,3].set_title("m / ‖e_quad‖ target ratio")
    axes[1,0].set_title("m / capacity-projected target")
    axes[1,1].set_title("2λm/‖g‖")
    axes[1,2].set_title("cos(d, -g)")
    axes[1,3].set_title("held-out accuracy")

    for ax in axes.flat:
        ax.set_xscale("log")
        ax.set_xlabel("λ")
        ax.grid(True, alpha=0.2)
    for ax in [axes[0,3], axes[1,0], axes[1,1], axes[1,2]]:
        ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
    axes[1,3].legend(loc="best", fontsize=7)
    plt.tight_layout()
    plt.show()

    # Seed-level slopes for m, energy norm, gradient norm, and target ratios.
    slope_rows = []
    for spec, df in seed_dfs.items():
        for metric in ["m_obs", "e_energy_norm", "g_norm", "m_over_m_quad", "m_over_m_subspace"]:
            vals = []
            for seed, sub in df[df["λ"] >= 0.1].groupby("seed"):
                sub = sub.sort_values("λ")
                if len(sub) >= 2 and np.all(np.asarray(sub[metric]) > 0):
                    vals.append(float(np.polyfit(np.log10(sub["λ"]), np.log10(sub[metric]), 1)[0]))
            mean, lo, hi = bootstrap_ci_mean(vals)
            slope_rows.append({
                "spec": spec, "metric": f"log {metric} slope vs log λ (λ≥0.1)",
                "mean": mean, "ci_low": lo, "ci_high": hi, "n_seeds": len(vals),
            })

    slope_df = pd.DataFrame(slope_rows)
    print("\nSeed-level slopes. If g_norm changes strongly with λ, the clean 1/λ sweep is confounded by changing task geometry.")
    display(slope_df)

    # Full target-vs-observed table with CIs.
    target_rows = []
    quantities = [
        ("m_obs", "median observed m"),
        ("e_energy_norm", "median observed energy-space ‖e‖"),
        ("g_norm", "median ‖g‖"),
        ("m_lin_target", "median ‖e_lin‖ target"),
        ("m_quad_target", "median ‖e_quad‖ target"),
        ("m_subspace_target", "median capacity-projected m target"),
        ("m_over_m_lin", "median m / ‖e_lin‖"),
        ("m_over_m_quad", "median m / ‖e_quad‖"),
        ("m_over_m_subspace", "median m / capacity target"),
        ("calib_linear", "median 2λm/‖g‖"),
        ("cos_d_neg_g", "median cos(d,-g)"),
        ("task_metric", "held-out accuracy"),
    ]
    for spec, df in seed_dfs.items():
        for lam, sub in df.groupby("λ"):
            for key, label in quantities:
                mean, lo, hi = bootstrap_ci_mean(sub[key].values)
                target_rows.append({
                    "spec": spec, "λ": lam, "quantity": label,
                    "mean": mean, "ci_low": lo, "ci_high": hi,
                    "n_seeds": len(sub),
                })

    target_df = pd.DataFrame(target_rows)
    print("\nTheoretical target versus observed values (seed-level 95% CIs):")
    display(target_df)

    compact = target_df[
        target_df["spec"].isin(["gauge_fixed_pre_head", "gauge_fixed_query_mlp", "gauge_fixed_pre_pullback", "gauge_fixed_post_wo"]) &
        target_df["quantity"].isin(["median observed m", "median observed energy-space ‖e‖",
                                    "median m / ‖e_quad‖",
                                    "median m / capacity target",
                                    "median cos(d,-g)", "held-out accuracy"])
    ]
    print("\nCompact gauge-fixed coordinate comparison:")
    display(compact)


## Optional experiment — fixed-backbone λ probe

The clean inverse-$\lambda$ law assumes the local task geometry is fixed while
$\lambda$ changes.  The main sweep retrains the entire model for each $\lambda$,
so $g$, $H$, and the feasible directions can drift.  This optional probe trains
a reference model once, freezes the representation and readout, and re-optimizes
only the magnitude channel for each $\lambda$.  It is intentionally off by
default; set `RUN_FIXED_BACKBONE_PROBE=True` in the hyperparameter cell to run it.


In [ ]:
def _set_controller_lambda(model, lam: float):
    model.lam = float(lam)
    model.attention.lam = float(lam)
    model.attention.controller.lam = float(lam)


def _magnitude_parameters(model):
    params = [model.attention.scalar_log_b]
    if model.attention.magnitude_net is not None:
        params += list(model.attention.magnitude_net.parameters())
    return params


def freeze_except_magnitude(model):
    mag_ids = {id(p) for p in _magnitude_parameters(model)}
    for p in model.parameters():
        p.requires_grad_(id(p) in mag_ids)


def run_fixed_backbone_probe(root: Path, lambdas, seeds: List[int],
                             lam_ref: float = LAM_FIXED_BACKBONE_REF,
                             steps_backbone: int = STEPS_5_2,
                             steps_mag: int = STEPS_FIXED_BACKBONE_MAG,
                             force: bool = FORCE_RERUN):
    testbed = "attention_5_2_fixed_backbone_post_wo"
    print("=== optional fixed-backbone post-WO λ probe ===")
    for seed in seeds:
        set_seed(seed)
        data = make_selective_copy_data(seed=seed)
        base = SelectiveCopyModel(V=V, L=L, K=K, d_model=D_MODEL, n_heads=N_HEADS,
                                  mode="gauge_fixed", lam=float(lam_ref),
                                  magnitude_mode="query_mlp", gauge_location="post_wo",
                                  penalty_metric="euclidean")
        _ = train_attention(base, data, steps=steps_backbone, log_every=steps_backbone)

        for lam in lambdas:
            meta = RunMeta(testbed=testbed, variant="gauge_fixed", lam=float(lam), seed=seed,
                           extra={"experiment": "fixed_backbone", "lam_ref": float(lam_ref),
                                  "gauge_location": "post_wo", "magnitude_mode": "query_mlp"})
            logger = RunLogger(root, meta, config={
                "experiment": "fixed_backbone", "lam_ref": float(lam_ref),
                "lam": float(lam), "seed": seed,
                "steps_backbone": steps_backbone, "steps_mag": steps_mag,
                "gauge_location": "post_wo", "magnitude_mode": "query_mlp",
                "gauge_lib_path": str(GAUGE_LIB_PATH),
                "gauge_lib_sha256": GAUGE_LIB_SHA256,
            })
            if logger.path.exists() and not force:
                continue

            model = deepcopy(base)
            _set_controller_lambda(model, float(lam))
            freeze_except_magnitude(model)
            train_log = train_attention(model, data, steps=steps_mag, log_every=100)
            diag = eval_diagnostics_attention(model, data, float(lam))
            for row in train_log:
                logger.log_step(**row)
            logger.update_final(**diag)
            logger.save()
        print(f"  seed={seed} done")


if RUN_FIXED_BACKBONE_PROBE:
    run_fixed_backbone_probe(ROOT, LAMBDAS_5_2, SEEDS)
else:
    print("Fixed-backbone probe skipped. Set RUN_FIXED_BACKBONE_PROBE=True to run it.")


## Experiment 5.3 — Residual decomposition and calibration failure modes

The 5.2 v3 runs contain full-Hessian `r_quad`, linearized `r_lin`, and both
absolute and normalized stationary residuals.  The normalized stationary
residual is important at large λ, where the absolute residual can grow simply
because the equilibrium forces are larger.


In [ ]:
# 5.3 — residual summary from 5.2 v3 runs
runs_by_spec = load_calibration_runs()
seed_dfs = {spec: seed_metric_rows_for_spec(runs)
            for spec, runs in runs_by_spec.items() if len(runs) > 0}

if not seed_dfs:
    print("No v3 calibration runs found. Run the 5.2 training cell above.")
else:
    resid_curve_rows = []
    for spec, df in seed_dfs.items():
        for lam, sub in df.groupby("λ"):
            resid_curve_rows.append({
                "spec": spec,
                "λ": lam,
                "r_lin": float(sub["r_lin"].mean()),
                "r_quad": float(sub["r_quad"].mean()),
                "r_subspace": float(sub["r_subspace"].mean()),
                "r_quad_over_r_lin": float(sub["r_quad_over_r_lin"].mean()),
                "r_subspace_over_r_quad": float(sub["r_subspace_over_r_quad"].mean()),
                "r_stat": float(sub["r_stat"].mean()),
                "r_stat_norm": float(sub["r_stat_norm"].mean()),
                "m_over_m_quad": float(sub["m_over_m_quad"].mean()),
                "m_over_m_subspace": float(sub["m_over_m_subspace"].mean()),
                "g_norm": float(sub["g_norm"].mean()),
            })
    resid_curves = pd.DataFrame(resid_curve_rows)

    fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))

    for spec, sub in resid_curves.groupby("spec"):
        label = spec.replace("_", " ")
        axes[0].loglog(sub["λ"], sub["r_lin"], "o--", alpha=0.45, label=f"{label}: r_lin")
        axes[0].loglog(sub["λ"], sub["r_quad"], "s-", label=f"{label}: r_quad")
        axes[1].loglog(sub["λ"], sub["r_subspace"], "o-", label=label)
        axes[2].semilogx(sub["λ"], sub["r_quad_over_r_lin"], "o-", label=label)
        axes[3].semilogx(sub["λ"], sub["r_stat_norm"], "o-", label=label)

    axes[0].set_title("linearized vs full-Hessian residual")
    axes[0].set_xlabel("λ"); axes[0].set_ylabel("median residual")
    axes[1].set_title("capacity-projected residual")
    axes[1].set_xlabel("λ"); axes[1].set_ylabel("median r_subspace")
    axes[2].set_title("curvature improvement")
    axes[2].set_xlabel("λ"); axes[2].set_ylabel("r_quad / r_lin")
    axes[2].axhline(1.0, color="grey", linestyle="--", linewidth=1)
    axes[3].set_title("normalized stationary residual")
    axes[3].set_xlabel("λ"); axes[3].set_ylabel("normalized residual")
    axes[3].axhline(0.0, color="grey", linestyle="--", linewidth=1)
    for ax in axes:
        ax.grid(True, alpha=0.2)
    axes[3].legend(loc="best", fontsize=6)
    plt.tight_layout()
    plt.show()

    resid_ci_rows = []
    for spec, df in seed_dfs.items():
        for lam, sub in df.groupby("λ"):
            for key in ["r_lin", "r_quad", "r_subspace", "r_quad_over_r_lin",
                        "r_subspace_over_r_quad", "r_stat", "r_stat_norm",
                        "m_over_m_quad", "m_over_m_subspace", "g_norm"]:
                mean, lo, hi = bootstrap_ci_mean(sub[key].values)
                resid_ci_rows.append({
                    "spec": spec, "λ": lam, "metric": key,
                    "mean": mean, "ci_low": lo, "ci_high": hi, "n_seeds": len(sub),
                })
    resid_ci_df = pd.DataFrame(resid_ci_rows)

    print("\nResidual decomposition and failure-mode diagnostics with seed-level 95% CIs:")
    display(resid_ci_df)

    print("\nInterpretation guide:")
    print("  • post_wo tests the actual residual-space magnitude m=||attention output||.")
    print("  • pre_pullback keeps the head-space gauge fix but penalizes ||W_O e||².")
    print("  • r_quad << r_lin: linear inverse-λ calibration is weakened by curvature.")
    print("  • r_subspace << r_quad: the observed head directions explain capacity-limited deviations.")
    print("  • g_norm changing with λ: independently retrained λ sweeps do not hold task geometry fixed.")


## Summary

A run-counts table you can sanity-check before moving on. The attention
testbeds used in the main story are:

- `attention_5_1` for gauge traversal and factor-only drift,
- `attention_5_2_v3_*` for λ calibration and residual decomposition.

`05_analysis.ipynb` glob-loads these by testbed name.

In [ ]:
# Summary of saved runs
for testbed_dir in sorted(ROOT.iterdir()):
    if not testbed_dir.is_dir(): continue
    runs_dir = testbed_dir / "runs"
    if not runs_dir.exists(): continue
    files = sorted(runs_dir.glob("*.pkl"))
    if not files: continue
    if "attention" in testbed_dir.name or "synthetic" in testbed_dir.name:
        print(f"{testbed_dir.name:<28} {len(files)} runs")
        for f in files[:2]:
            print(f"    {f.name}")
        if len(files) > 2:
            print(f"    ... ({len(files) - 2} more)")